In [1]:
import torch
import torch.nn as nn

torch.manual_seed(1)

In [2]:
G = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Linear(256, 784),
    nn.Tanh()
)


In [3]:
D = nn.Sequential(
    nn.Linear(784, 256),
    nn.LeakyReLU(0.2),
    nn.Linear(256, 128),
    nn.LeakyReLU(0.2),
    nn.Linear(128, 1)
)

In [4]:
z = torch.randn(8, 64)

In [5]:
fake = G(z)

In [6]:
real = torch.randn(8, 784)

In [7]:
alpha = torch.rand(8, 1)
interpolated = alpha*real + (1-alpha)*fake
interpolated.requires_grad_(True)

critic_output = D(interpolated)

In [8]:
gradient = torch.autograd.grad(
    outputs=critic_output,
    inputs=interpolated,
    grad_outputs=torch.ones_like(critic_output),
    create_graph=True
)[0]

gradient = gradient.view(8, -1)

gradient_norm = torch.sqrt(
    torch.sum(gradient**2, dim=1) + 1e-8
)

In [9]:
gradient_penalty = torch.mean((gradient_norm - 1)**2)


In [10]:
critic_loss = D(fake.detach()).mean() - D(real).mean()
critic_loss = critic_loss + 10*gradient_penalty

In [11]:
generator_loss = -D(fake).mean()

print("Gradient Penalty:", gradient_penalty.item())
print("Critic Loss:", critic_loss.item())
print("Generator Loss:", generator_loss.item())

Gradient Penalty: 0.8058078289031982
Critic Loss: 8.085357666015625
Generator Loss: -0.08994343131780624
